# DNS 모집단 정책 검증

## tl;dr

DNS 605경주 중 1두 DNS는 555경주, 2두는 46경주, 3두는 4경주로 99.34%가 1~2두 취소였다. 모든 경주가 완료 상태·7개 공식 승식 완전·1착 존재·공식 연승 조합 정상 조건을 만족했다. DNS 말만 모델 행에서 제외하고 DNS 존재 자체로 경주를 제외하지 않는 정책을 지지한다. 이 Notebook은 DuckDB를 읽기 전용으로 조회하며 Feature Snapshot을 생성하지 않는다.

## Context & Methods

대상은 완료 상태이고 공식 연승 적중 원문이 존재하며 `result_status='DNS'`인 말이 한 마리 이상 있는 경주다. DNS 수, 정상 완주·주행정지·실격 수, 공식 승식 수, 연승 적중마 결합과 경주 상태를 독립적으로 검사한다.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'PROJECT_CHARTER.md').exists():
    project_root = project_root.parent
database_path = project_root / 'data' / 'warehouse' / 'kra.duckdb'
connection = duckdb.connect(str(database_path), read_only=True)
print(f'Source: {database_path}')

Source: C:\Users\jjk00\Documents\GitHub\kra-racing-analytics\data\warehouse\kra.duckdb


## Data

### 1. 경주별 DNS 프로파일

In [2]:
race_profile = connection.sql("""
WITH plc_races AS (
    SELECT DISTINCT race_id FROM canonical.winning_payout
    WHERE pool_code = '연식' AND parse_status = 'PARSED'
), runner_profile AS (
    SELECT r.race_id, r.race_date, r.meet_name, r.race_no, r.race_status,
           r.runner_count AS registered_runners, count(*) AS result_rows,
           count(*) FILTER (WHERE rr.result_status = 'DNS') AS dns_count,
           count(*) FILTER (WHERE rr.is_valid_start) AS valid_starters,
           count(*) FILTER (WHERE rr.result_status = 'FINISHED') AS finishers,
           count(*) FILTER (WHERE rr.result_status = 'RACE_STOPPED') AS stopped_count,
           count(*) FILTER (WHERE rr.result_status = 'DISQUALIFIED') AS disqualified_count,
           count(*) FILTER (WHERE NOT rr.is_valid_start AND rr.result_status != 'DNS') AS other_nonstart_count,
           count(*) FILTER (WHERE rr.official_finish_rank = 1) AS rank1_count
    FROM canonical.race r JOIN canonical.runner_result rr USING (race_id)
    JOIN plc_races p USING (race_id)
    WHERE r.race_status = 'COMPLETED'
    GROUP BY ALL
), sales_profile AS (
    SELECT race_id, count(DISTINCT pool_code) AS sales_pool_count
    FROM canonical.sales_dividend GROUP BY race_id
), plc_profile AS (
    SELECT race_id, count(*) AS plc_positive_count
    FROM canonical.winning_payout
    WHERE pool_code = '연식' AND parse_status = 'PARSED' GROUP BY race_id
)
SELECT rp.*, sp.sales_pool_count, pp.plc_positive_count
FROM runner_profile rp JOIN sales_profile sp USING (race_id) JOIN plc_profile pp USING (race_id)
WHERE dns_count > 0 ORDER BY race_date, meet_name, race_no
""").df()
print(f'Races: {len(race_profile):,}')
display(race_profile.head())

Races: 605


,race_id,race_date,meet_name,race_no,race_status,registered_runners,result_rows,dns_count,valid_starters,finishers,stopped_count,disqualified_count,other_nonstart_count,rank1_count,sales_pool_count,plc_positive_count
0,2024-01-06|1|R03,2024-01-06,서울,3,COMPLETED,10,10,1,9,8,1,0,0,1,7,3
1,2024-01-06|1|R08,2024-01-06,서울,8,COMPLETED,11,11,2,9,9,0,0,0,1,7,3
2,2024-01-07|3|R01,2024-01-07,부산경남,1,COMPLETED,11,11,1,10,10,0,0,0,1,7,3
3,2024-01-07|1|R05,2024-01-07,서울,5,COMPLETED,9,9,1,8,8,0,0,0,1,7,3
4,2024-01-07|1|R06,2024-01-07,서울,6,COMPLETED,10,10,1,9,9,0,0,0,1,7,3


## Results

### 2. DNS 마릿수 분포

In [3]:
dns_distribution = (race_profile.groupby('dns_count', as_index=False)
                    .agg(races=('race_id', 'count'),
                         runner_rows=('result_rows', 'sum'),
                         valid_starters=('valid_starters', 'sum')))
dns_distribution['race_share'] = dns_distribution['races'] / len(race_profile)
display(dns_distribution)

normal_cancel_summary = pd.DataFrame({
    'metric': ['one_or_two_dns_races', 'one_or_two_dns_share', 'max_dns_count',
               'median_valid_starters', 'min_valid_starters'],
    'value': [int((race_profile['dns_count'] <= 2).sum()),
              float((race_profile['dns_count'] <= 2).mean()),
              int(race_profile['dns_count'].max()),
              float(race_profile['valid_starters'].median()),
              int(race_profile['valid_starters'].min())]
})
display(normal_cancel_summary)

,dns_count,races,runner_rows,valid_starters,race_share
0,1,555,6011,5456,0.917355
1,2,46,496,404,0.076033
2,3,4,44,32,0.006612


,metric,value
0,one_or_two_dns_races,601.000000
1,one_or_two_dns_share,0.993388
2,max_dns_count,3.000000
3,median_valid_starters,10.000000
4,min_valid_starters,6.000000


### 3. 취소·비정상 진행 신호

In [4]:
status_summary = pd.DataFrame({
    'check': [
        'race_status_not_completed', 'incomplete_official_pools',
        'runner_count_mismatch', 'valid_plus_dns_mismatch',
        'missing_first_place', 'invalid_plc_positive_count',
        'other_nonstart_status', 'race_with_stopped_runner',
        'race_with_disqualified_runner'
    ],
    'races': [
        int((race_profile['race_status'] != 'COMPLETED').sum()),
        int((race_profile['sales_pool_count'] != 7).sum()),
        int((race_profile['registered_runners'] != race_profile['result_rows']).sum()),
        int(((race_profile['valid_starters'] + race_profile['dns_count']) != race_profile['result_rows']).sum()),
        int((race_profile['rank1_count'] == 0).sum()),
        int((~race_profile['plc_positive_count'].between(2, 4)).sum()),
        int((race_profile['other_nonstart_count'] > 0).sum()),
        int((race_profile['stopped_count'] > 0).sum()),
        int((race_profile['disqualified_count'] > 0).sum())
    ]
})
display(status_summary)

all_dns_by_race_status = connection.sql("""
SELECT r.race_status, count(*) AS dns_rows, count(DISTINCT rr.race_id) AS races
FROM canonical.runner_result rr JOIN canonical.race r USING (race_id)
WHERE rr.result_status = 'DNS' GROUP BY r.race_status ORDER BY r.race_status
""").df()
display(all_dns_by_race_status)

review_races = race_profile.loc[
    (race_profile['dns_count'] > 2)
    | (race_profile['stopped_count'] > 0)
    | (race_profile['disqualified_count'] > 0)
    | (race_profile['sales_pool_count'] != 7)
    | (race_profile['rank1_count'] == 0),
    ['race_id', 'race_date', 'meet_name', 'race_no', 'registered_runners', 'dns_count',
     'valid_starters', 'finishers', 'stopped_count', 'disqualified_count',
     'sales_pool_count', 'plc_positive_count']
]
print(f'Review-list races: {len(review_races):,}')
display(review_races)

,check,races
0,race_status_not_completed,0
1,incomplete_official_pools,0
2,runner_count_mismatch,0
3,valid_plus_dns_mismatch,0
4,missing_first_place,0
5,invalid_plc_positive_count,0
6,other_nonstart_status,0
7,race_with_stopped_runner,13
8,race_with_disqualified_runner,0


,race_status,dns_rows,races
0,COMPLETED,659,605


Review-list races: 17


,race_id,race_date,meet_name,race_no,registered_runners,dns_count,valid_starters,finishers,stopped_count,disqualified_count,sales_pool_count,plc_positive_count
0,2024-01-06|1|R03,2024-01-06,서울,3,10,1,9,8,1,0,7,3
21,2024-02-17|1|R05,2024-02-17,서울,5,9,1,8,7,1,0,7,3
31,2024-03-02|1|R09,2024-03-02,서울,9,11,2,9,8,1,0,7,3
60,2024-04-13|1|R08,2024-04-13,서울,8,10,1,9,8,1,0,7,3
89,2024-05-18|1|R03,2024-05-18,서울,3,11,3,8,8,0,0,7,3
98,2024-05-26|1|R02,2024-05-26,서울,2,11,3,8,8,0,0,7,3
131,2024-07-12|3|R07,2024-07-12,부산경남,7,11,1,10,9,1,0,7,3
174,2024-09-29|1|R10,2024-09-29,서울,10,11,3,8,8,0,0,7,3
176,2024-10-04|3|R02,2024-10-04,부산경남,2,11,1,10,9,1,0,7,3
180,2024-10-06|3|R04,2024-10-06,부산경남,4,11,3,8,8,0,0,7,3


### 4. 기간·경마장 분포

In [5]:
temporal_profile = (race_profile.assign(year=pd.to_datetime(race_profile['race_date']).dt.year)
                    .groupby(['year', 'meet_name'], as_index=False)
                    .agg(races=('race_id', 'count'), dns_rows=('dns_count', 'sum')))
display(temporal_profile)

,year,meet_name,races,dns_rows
0,2024,부산경남,68,76
1,2024,서울,164,183
2,2025,부산경남,65,68
3,2025,서울,166,183
4,2026,부산경남,31,31
5,2026,서울,111,118


## Takeaways

- DNS 1두 555경주, 2두 46경주, 3두 4경주로 601경주(99.34%)가 1~2두 취소였다.
- DNS가 있는 모든 경주에서 경주 상태 완료, 7개 공식 승식, 등록·결과 행 일치, 1착과 2~4개 공식 연승 적중마를 확인했다.
- 3두 DNS 4경주도 각각 유효 출전마 8두와 정상 완주 결과가 있어 경주 단위 제외 사유가 아니다.
- 주행정지 말이 함께 있는 13경주는 공식 결과와 승식이 정상이며, 해당 말은 실제 출전한 음성 행으로 보존한다.
- 정책은 DNS 말 단위 제외를 기본으로 하고, 경주 단위 제외는 취소·결과 미확정·공식 연승 누락·타깃 결합 실패·미해결 비출전 상태처럼 경주 전체의 학습 계약이 깨질 때만 적용한다.

In [6]:
connection.close()